In [0]:
%sql
-- ============================================================
-- ETL: bronze.categorias_raw → silver.categorias
-- Patrón: CREATE OR REPLACE (one-time, tabla estática)
-- No usamos MERGE porque son 180 filas que no cambian.
-- Si Meli agrega categorías nuevas, se recorre el script
-- load_categorias.py y se vuelve a correr este ETL.
-- ============================================================

CREATE OR REPLACE TABLE iniciacion_deportiva.silver.categorias AS

WITH path_expandido AS (
    SELECT
        TRIM(categoria_id)  AS categoria_id,
        TRIM(nombre)        AS nombre,
        FROM_JSON(
            path_from_root,
            'ARRAY<STRUCT<id:STRING, name:STRING>>'
        )                   AS path
    FROM iniciacion_deportiva.bronze.categorias_raw
    WHERE categoria_id IS NOT NULL AND TRIM(categoria_id) != ''
)

SELECT
    categoria_id,
    nombre,
    TRIM(GET(path, 0).name)                        AS nivel_1,
    TRIM(GET(path, 1).name)                        AS nivel_2,
    TRIM(GET(path, 2).name)                        AS nivel_3,
    TRIM(GET(path, 3).name)                        AS nivel_4,
    TRIM(GET(path, 4).name)                        AS nivel_5,
    ARRAY_JOIN(
        TRANSFORM(path, x -> TRIM(x.name)), ' > '
    )                                              AS path_completo,
    'iniciacion_deportiva.bronze.categorias_raw'   AS _source_table,
    CURRENT_TIMESTAMP()                            AS _processing_timestamp
FROM path_expandido
ORDER BY categoria_id;